In [25]:
!pip install -q --upgrade youtube-transcript-api langchain langchain-google-genai langchain-community faiss-cpu python-dotenv langchain-huggingface

In [41]:
import os
from google.colab import userdata
from langchain_huggingface import HuggingFaceEndpoint, HuggingFaceEmbeddings,ChatHuggingFace
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("✅ Hugging Face token loaded from Colab secrets")
except userdata.SecretNotFoundError:
    print("❌ HF_TOKEN not found in Colab secrets. Please add it.")
except Exception as e:
    print(f"❌ Error loading HF_TOKEN: {e}")

# 1. Setup the LLM
llm_endpoint = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    task="conversational",
    max_new_tokens=512,
    do_sample=False
)
llm=ChatHuggingFace(llm=llm_endpoint)

# 2. Setup the Embedding Model
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5"
)

✅ Hugging Face token loaded from Colab secrets


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [28]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

print("✅ All imports successful!")

✅ All imports successful!


In [46]:
from langchain_community.document_loaders import YoutubeLoader

video_url = "https://www.youtube.com/watch?v=vXVRs_Sar6Y&t=395s"

try:
    # LangChain handles the video ID extraction and API calls for you
    loader = YoutubeLoader.from_youtube_url(
        video_url,
        add_video_info=False, # Set to True if you also want the title/author
        language=["en"]
    )

    docs = loader.load()
    transcript = docs[0].page_content

    print(f"✅ Transcript fetched successfully!")
    print(f"📜 Length: {len(transcript):,} characters | ~{len(transcript.split()):,} words")
    print(f"\n📖 Preview:\n{transcript[:500]}...")

except Exception as e:
    print(f"❌ Error fetching transcript: {e}")

✅ Transcript fetched successfully!
📜 Length: 17,551 characters | ~3,099 words

📖 Preview:
Hello, friends! If you remember, in 2014, there was a blockbuster film by Director Christopher Nolan, Interstellar. In this film, space-related concepts, such as wormholes, black holes, alien planets, were depicted in a scientifically accurate way. But perhaps the most shocking scene,
was at the end of the film. When in the climax, the main character of the film Cooper falls into a Black Hole. In the film, the name of the black hole was Gargantua. Cooper falls into the black hole with his spacec...


In [47]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = splitter.create_documents([transcript])

print(f"✅ Split into {len(chunks)} chunks")
print(f"\n📄 Sample chunk (chunk #0):\n{chunks[0].page_content[:300]}...")

✅ Split into 27 chunks

📄 Sample chunk (chunk #0):
Hello, friends! If you remember, in 2014, there was a blockbuster film by Director Christopher Nolan, Interstellar. In this film, space-related concepts, such as wormholes, black holes, alien planets, were depicted in a scientifically accurate way. But perhaps the most shocking scene,...


In [48]:
vector_store = FAISS.from_documents(chunks, embeddings)

print(f"✅ Vector store created with {len(chunks)} document embeddings using HuggingFace")

✅ Vector store created with 27 document embeddings using HuggingFace


In [49]:
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={"k": 4}
)

# Test the retriever
test_query = "What is this video about?"
test_results = retriever.invoke(test_query)

print(f"✅ Retriever ready! Test query: '{test_query}'")
print(f"📄 Retrieved {len(test_results)} chunks")
for i, doc in enumerate(test_results):
    print(f"\n--- Chunk {i+1} (first 150 chars) ---")
    print(doc.page_content[:150])

✅ Retriever ready! Test query: 'What is this video about?'
📄 Retrieved 4 chunks

--- Chunk 1 (first 150 chars) ---
concept of 5-Dimension mentioned in Intersettlar, is another interesting concept. Let's talk about it in a future video. That's all for now. If you li

--- Chunk 2 (first 150 chars) ---
Hello, friends! If you remember, in 2014, there was a blockbuster film by Director Christopher Nolan, Interstellar. In this film, space-related concep

--- Chunk 3 (first 150 chars) ---
was at the end of the film. When in the climax, the main character of the film Cooper falls into a Black Hole. In the film, the name of the black hole

--- Chunk 4 (first 150 chars) ---
of the spinning particles. The area that is brighter, is coming towards us, and the dim area, is going away from us. Coming back to the image from the


## 8️⃣ Set Up LLM & Prompt Template

In [50]:
prompt = PromptTemplate(
    template="""You are a helpful assistant that answers questions based on YouTube video transcripts.
Answer only from the provided transcript context. If the context is insufficient, say "I don't have enough information from the video to answer that question."

Context from the video transcript:
{context}

Question: {question}

Answer:""",
    input_variables=['context', 'question']
)

print("✅ LLM (HuggingFace) and prompt template configured")

✅ LLM (HuggingFace) and prompt template configured


## 9️⃣ Manual Step-by-Step Invocation

Let's manually walk through the retrieval → prompt → generation pipeline.

In [51]:
# Step 1: Define your question
question = "What are the main topics discussed in this video?"

# Step 2: Retrieve relevant chunks
retrieved_docs = retriever.invoke(question)
print(f"📄 Retrieved {len(retrieved_docs)} relevant chunks\n")

# Step 3: Format the context
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
print(f"📝 Context length: {len(context_text)} characters\n")

# Step 4: Create the final prompt
final_prompt = prompt.invoke({"context": context_text, "question": question})

# Step 5: Generate the answer
answer = llm.invoke(final_prompt)

print("🤖 Answer:")
print(answer.content)

📄 Retrieved 4 relevant chunks

📝 Context length: 1617 characters

🤖 Answer:
According to the video transcript, the main topic discussed is the concept of the 5-Dimension mentioned in Interstellar, specifically in the context of a black hole. Additionally, the video seems to be an introduction to a video about black holes, as hinted by the title "Black".


In [52]:
def format_docs(retrieved_docs):
    """Join retrieved document page contents with double newlines."""
    return "\n\n".join(doc.page_content for doc in retrieved_docs)


# Build the parallel chain
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

# Output parser
parser = StrOutputParser()

# Full RAG chain
main_chain = parallel_chain | prompt | llm | parser

print("✅ RAG chain assembled!")

✅ RAG chain assembled!


In [53]:
# Test with a single question
answer = main_chain.invoke("Tell me about the video")

print("🤖 Answer:")
print(answer)

🤖 Answer:
The video is about the concept of a black hole and its depiction in the movie "Interstellar" directed by Christopher Nolan. It discusses the visuals of a black hole, including the accretion disk, which creates an optical illusion due to gravity, and the photonsphere, which is the last circle of light that orbit the black hole. The video also touches on the idea of a five-dimensional space and tesseract, as seen in the movie.


In [54]:
while True:
    user_question = input("\n🧑 You: ")

    if user_question.lower().strip() in ['quit', 'exit', 'q']:
        print("\n👋 Goodbye!")
        break

    if not user_question.strip():
        print("⚠️  Please enter a question.")
        continue

    try:
        answer = main_chain.invoke(user_question)
        print(f"\n🤖 Bot: {answer}")
    except Exception as e:
        print(f"\n❌ Error: {e}")

💬 YouTube Chatbot — Interactive Mode
Ask any question about the video.
Type 'quit' or 'exit' to stop.


🧑 You: who is the person shown in the video

🤖 Bot: According to the video transcript, the person shown in the video is not explicitly mentioned. The narrator is referring to a character from the movie Interstellar, specifically Cooper, the main character, who falls into a black hole, specifically Gargantua.

🧑 You: tell me in short about what the video talks about

🤖 Bot: The video talks about the concept of the 5-Dimensional space, specifically the "tesseract" mentioned in the 2014 film Interstellar, directed by Christopher Nolan.

🧑 You: quit

👋 Goodbye!
